# Movement-strategy validation

**Question.** How faithfully does each movement strategy reproduce a commanded
path, and when two strategies differ, which ingredient caused it?

**Scope.** Standalone; the control strategy only. Everything is in *controller
units* (command counts). Actuator calibration - servo ticks, spool radius, dead
zone, latency - is out of scope and belongs to `validetion/Servomotor` and
`validetion/Servo+thimble`.

## The four strategies

All four are run exactly as the device defines them. Each is built from two
independent ingredients (`motor_controller.STRATEGY_DEFINITIONS`):

| strategy | direction quantisation | kinematic model |
|---|---|---|
| `CARDINAL` | 4-way | planar |
| `CARDINAL_DIAGONAL` | 8-way | planar |
| `FREE_FORM` | none | planar |
| `IK` | none | 3-D mechanism |

Writing the ingredients out matters for interpretation: `CARDINAL_DIAGONAL` and
`IK` differ in **both**, so a difference measured between just those two cannot
be attributed to the quantisation or to the model. The ingredient grid below
resolves that by running every combination, including the two no named strategy
occupies.

**Limitation.** Each model is decoded by its own inverse - trilateration for
planar, wire FK for the mechanism - so this measures strategy-induced path
error, not physical mechanism accuracy. That FK genuinely inverts the
controller's IK path is verified in `tests/test_wire_forward_kinematics.py`.

All computation lives in `analysis.py` and `figures.py`; this notebook only
calls them, so there is exactly one implementation.

In [1]:
import pandas as pd

from analysis import (
    StudyConfig, ingredient_grid, named_strategy_table, run_study, strategy_positions,
)
from figures import (
    plot_ingredient_grids, plot_motor_commands, plot_reconstructed_paths,
    plot_strategy_comparison,
)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

config = StudyConfig()
samples, metrics = run_study(config)

print(f"radius {config.radius:.0f} controller units, "
      f"{config.line_steps + config.circle_steps + config.return_steps + 1} samples per run")
strategy_positions(config)

radius 160 controller units, 301 samples per run


,quantisation,kinematic_model
strategy,,
cardinal,cardinal_4,planar
cardinal_diagonal,cardinal_8,planar
free_form,none,planar
ik,none,ik


## Results per strategy

`quantisation_rms` is what the direction ingredient costs; `execution_rms` is
what the model ingredient plus integer command truncation cost.

In [2]:
named_strategy_table(metrics, config).round(3)

,quantisation,kinematic_model,quantisation_rms,execution_rms,total_rms,rms_command,max_step_jump
run,,,,,,,
cardinal,cardinal_4,planar,61.067,0.706,60.962,100.168,223.0
cardinal_diagonal,cardinal_8,planar,32.644,0.769,32.589,100.273,121.0
free_form,none,planar,0.000,0.824,0.824,100.318,5.0
ik,none,ik,0.000,3.517,3.517,26.365,2.0


## Attributing a difference to one ingredient

Every (quantisation, model) combination, including the two cells no named
strategy uses. Read **down a column** for the quantisation effect at a fixed
model; read **across a row** for the model effect at a fixed quantisation.

In [3]:
print("Total RMS error")
ingredient_grid(metrics, "total_rms").round(2)

Total RMS error


kinematic_model,ik,planar
quantisation,,
cardinal_4,60.87,60.96
cardinal_8,32.61,32.59
none,3.52,0.82


In [4]:
print("RMS motor command")
ingredient_grid(metrics, "rms_command").round(2)

RMS motor command


kinematic_model,ik,planar
quantisation,,
cardinal_4,26.48,100.17
cardinal_8,26.40,100.27
none,26.36,100.32


Error is set almost entirely by the quantisation ingredient (it changes
down the rows, barely across the columns). Command amplitude is set almost
entirely by the model ingredient (the reverse). So the amplitude difference
between `CARDINAL_DIAGONAL` and `IK` comes from the kinematic model, and their
path-error difference comes from the direction quantisation.

## Shape metrics, and why they mislead

Quantisation is radius-preserving, so every reconstructed point sits on the
commanded radius. Aspect ratio and circle-fit RMS therefore score all four
strategies as near-perfect circles while point-to-point error differs by more
than an order of magnitude. That is why point-to-point error is the primary
metric here.

In [5]:
names = [s.value for s in config.strategies]
metrics.loc[names, ["radial_rms", "circle_fit_rms", "aspect_ratio",
                    "closure_error", "decode_invalid_steps"]].round(3)

,radial_rms,circle_fit_rms,aspect_ratio,closure_error,decode_invalid_steps
run,,,,,
cardinal,0.602,0.108,1.001,2.687,0
cardinal_diagonal,0.699,0.161,1.001,2.687,0
free_form,0.745,0.257,1.001,0.000,0
ik,3.143,1.693,0.997,0.000,0


## Figures

In [6]:
for plot in (plot_reconstructed_paths(samples, config),
             plot_strategy_comparison(metrics, config),
             plot_ingredient_grids(metrics, config),
             plot_motor_commands(samples, config)):
    print("saved", plot.name)

saved reconstructed_paths_by_strategy.png
saved strategy_comparison.png
saved ingredient_grids.png
saved motor_commands_by_strategy.png
